[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/othnielObasi/airg-cookbook/blob/main/use_cases/19_pii_scanner_redaction.ipynb)

# 🔍 PII Scanner & Redaction — Governed Healthcare Pipeline

A healthcare support agent that accesses patient records. AIRG scans every tool call for PII
(SSNs, emails, credit cards, phone numbers) and automatically boosts risk scores to prevent data leakage.

**What you'll learn:**
- Scan tool arguments for PII entities with confidence scores
- See how PII detection integrates into the risk evaluation pipeline
- Field-level detection across nested tool arguments
- Real-world: preventing a support bot from emailing patient SSNs

**Setup:** Create an API key in your registered AIRG account, then set `GOVERNOR_API_KEY` in Colab Secrets (key icon) or as an environment variable.

## 1. Install & Configure

In [ ]:
!pip install -q httpx

import os, httpx

try:
    from google.colab import userdata
    API_KEY = userdata.get("GOVERNOR_API_KEY")
except Exception:
    API_KEY = os.getenv("GOVERNOR_API_KEY")

if not API_KEY:
    raise RuntimeError("Missing GOVERNOR_API_KEY. Create an API key in your registered AIRG account and add it to Colab Secrets or your environment.")

BASE = os.getenv("GOVERNOR_URL", "https://api.airg.nov-tia.com")
headers = {"X-API-Key": API_KEY, "Content-Type": "application/json"}
print(f"Connected to {BASE}")


## 2. Scenario — Support Bot Tries to Email Patient Data

In [ ]:
# A healthcare bot is about to email patient info; scan for PII before sending.
payload = {
    "tool": "send_email",
    "args": {
        "to": "billing@hospital.com",
        "subject": "Patient Record Transfer",
        "body": (
            "Patient: Jane Smith\n"
            "SSN: 987-65-4321\n"
            "DOB: 03/15/1985\n"
            "Insurance ID: BCBS-44112233\n"
            "Email: jane.smith@gmail.com\n"
            "Phone: (555) 867-5309\n"
            "CC on file: 4111-1111-1111-1111"
        ),
    },
}

r = httpx.post(f"{BASE}/pii/scan", headers=headers, json=payload)
r.raise_for_status()
scan = r.json()

print(f"PII detected: {scan.get('has_pii')}")
print(f"Total findings: {scan.get('total_findings', 0)}")
print(f"Risk boost: +{scan.get('risk_boost', 0)} points\n")

input_scan = scan.get("input_scan") or {}
for finding in input_scan.get("findings") or []:
    print(f"- {finding.get('entity_type', 'unknown'):20s} | {finding.get('value_redacted', '[redacted]')}")
    print(f"  Confidence: {finding.get('confidence', 'n/a')}  Field: {finding.get('field_path', 'n/a')}")


## 3. PII Boosts Risk in the Evaluation Pipeline

In [ ]:
# Now evaluate the same action through the full governance pipeline.
# PII detection should boost the risk score and may trigger review/block depending on policy.
r = httpx.post(f"{BASE}/actions/evaluate", headers=headers, json={
    "agent_id": "healthcare-bot",
    "tool": "send_email",
    "args": {"body": "Patient SSN: 987-65-4321, CC: 4111-1111-1111-1111"},
})
r.raise_for_status()
d = r.json()

print(f"Decision  : {d.get('decision')}")
print(f"Risk score: {d.get('risk_score')}/100")
print(f"Reason    : {d.get('explanation', 'N/A')}")

if d.get("decision") == "allow":
    print("WARNING: This healthcare PII send was allowed; policy should be tightened to review or block SSN/card exfiltration.")


## 4. Clean Data — No PII Detected

In [ ]:
# Compare with a clean internal note: no recipient email, no patient data, no payment data.
r = httpx.post(f"{BASE}/pii/scan", headers=headers, json={
    "tool": "internal_note",
    "args": {
        "body": "Reminder: Staff meeting at 3pm in Conference Room B.",
    },
})
r.raise_for_status()
clean = r.json()

print(f"PII detected: {clean.get('has_pii')}")
print(f"Findings    : {clean.get('total_findings', 0)}")
print(f"Risk boost  : +{clean.get('risk_boost', 0)} points")

if clean.get("has_pii"):
    print("WARNING: Clean internal note was flagged; inspect findings for false positives.")
else:
    print("Clean internal note passes with zero PII risk.")


## 5. Batch Scan — Multiple Messages

In [ ]:
# Scan multiple messages to find which ones contain PII.
messages = [
    {"tool": "chat", "args": {"text": "Hello, how can I help you?"}},
    {"tool": "chat", "args": {"text": "Your account 4532-1111-2222-3333 is active"}},
    {"tool": "chat", "args": {"text": "Call us at (800) 555-0199"}},
    {"tool": "chat", "args": {"text": "The weather is nice today"}},
]

print("Batch PII Scan Results:")
print("-" * 50)
for i, msg in enumerate(messages, start=1):
    r = httpx.post(f"{BASE}/pii/scan", headers=headers, json=msg)
    r.raise_for_status()
    s = r.json()
    label = "PII" if s.get("has_pii") else "clean"
    print(f"  Message {i}: {label}, {s.get('total_findings', 0)} findings, "
          f"risk_boost=+{s.get('risk_boost', 0)}")


## Summary

| Feature | What It Does |
|---|---|
| **PII Scan** | Detects SSNs, emails, credit cards, phone numbers |
| **Field-level** | Pinpoints which field contains PII |
| **Risk Boost** | Auto-increases risk score when PII found |
| **Pipeline Integration** | PII findings flow into evaluate() decisions |

→ [Full API docs](https://api.airg.nov-tia.com/docs) · [More recipes](https://github.com/othnielObasi/airg-cookbook)